<a href="https://colab.research.google.com/github/pajola/secai-course/blob/main/module-02-data-security/03_provenance_and_integrity.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" width="150"></a>

# 3 · Provenance and integrity — where did this data come from, and has anyone changed it?

**SecAI course · Data Security for AI, part 3**

The Northvale dataset arrived in three ingestion batches. Before training, the reviewer must be able to answer two questions for every row:
**where did it come from, and on what legal basis** (provenance), and **is it exactly what the source sent** (integrity)?
A row that fails either question is a liability: it may be unlawful to use, or it may have been poisoned (module 1, notebook 3: backdoors).

| | Data poisoning through the pipeline |
|---|---|
| **Goal** | make the trained model approve a profile it should reject (here: weak loan applications), without hurting overall accuracy |
| **Knowledge** | the data format and where the batches are stored; nothing about the model |
| **Capability** | write access to one batch **between export and training**: an insider, a compromised storage bucket, a malicious data vendor |

**Open-source tools used here:** [pandera](https://pandera.readthedocs.io) (schema and data-quality checks), Python's `hashlib` (SHA-256, what tools such as DVC
and Git LFS use under the hood), and [`cryptography`](https://cryptography.io) (Ed25519 signatures).

In [1]:
# Setup — on Google Colab, fetch the dataset generator and install the open-source tools used below.
import os, sys, urllib.request
if "google.colab" in sys.modules:
    if not os.path.exists("northvale.py"):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/pajola/secai-course/main/module-02-data-security/northvale.py", "northvale.py")
    %pip install -q faker pandera

In [2]:
import numpy as np, pandas as pd
from northvale import load, batch_sha256, reexport_from_source, REF_DATE
pd.set_option("display.max_colwidth", 60)

df, manifest = load()
manifest

,batch_id,origin,owner,legal_basis,license,rows,sha256
0,B1_core_banking,"Core banking export, applications 2022-2023",Credit Risk Office,GDPR Art. 6(1)(b) contract; DPIA-2024-07,internal,528,cfe5ae47c964e0fde1d41a9272f413a44a783a2959b09d040e840f3d...
1,B2_core_banking,"Core banking export, applications 2024",Credit Risk Office,GDPR Art. 6(1)(b) contract; DPIA-2024-07,internal,414,e0677aa080868b191fe0cb0e7c92d6e10431c84dcf1a0340b01ecb16...
2,B3_broker_purchase,"Lead list bought from a broker (LeadMax s.r.l., fictional)",,,,258,


The **provenance manifest** records, for each batch, where it came from, who owns it, the legal basis for processing it, and the SHA-256
checksum computed **when the batch left its source system**.

## 3.1 How a checksum detects changes

A cryptographic hash maps any input to a fixed-size fingerprint. Change one character and the fingerprint changes completely,
and nobody can construct a different dataset with the same fingerprint. `batch_sha256` hashes a batch in a canonical form (fixed column order, rows sorted by id).

In [3]:
b1 = df[df.source_batch == "B1_core_banking"]
edited = b1.copy(); edited.iloc[0, edited.columns.get_loc("approved")] ^= 1     # flip ONE label out of 528 rows
print("B1 as delivered :", batch_sha256(b1))
print("one label flipped:", batch_sha256(edited))

B1 as delivered : cfe5ae47c964e0fde1d41a9272f413a44a783a2959b09d040e840f3d800b787c
one label flipped: 0dcae503e0ea7b745fc71a885b2ad97cf3d99c86acfa262dbb1c1a85f13beaaa


## 3.2 Provenance check

For each batch: do the rows match the manifest, is there an owner and a legal basis, and does the checksum still match?

In [4]:
rows = []
for m in manifest.itertuples():
    batch = df[df.source_batch == m.batch_id]
    if not m.sha256:              integrity = "NO CHECKSUM"
    elif batch_sha256(batch) == m.sha256: integrity = "OK"
    else:                         integrity = "MISMATCH"
    rows.append({"batch": m.batch_id, "rows": len(batch), "declared rows": int(m.rows), "owner": m.owner or "—",
                 "legal basis": m.legal_basis or "—", "integrity": integrity})
prov = pd.DataFrame(rows).set_index("batch")
prov

,rows,declared rows,owner,legal basis,integrity
batch,,,,,
B1_core_banking,528,528,Credit Risk Office,GDPR Art. 6(1)(b) contract; DPIA-2024-07,OK
B2_core_banking,414,414,Credit Risk Office,GDPR Art. 6(1)(b) contract; DPIA-2024-07,MISMATCH
B3_broker_purchase,258,258,—,—,NO CHECKSUM


In [5]:
n = len(df)
no_origin = prov[(prov.owner == "—") | (prov["legal basis"] == "—")].rows.sum()
unverified = prov[prov.integrity == "MISMATCH"].rows.sum()
print(f"rows with no defensible origin (no owner / legal basis): {no_origin:4d}  ({no_origin / n:.1%})")
print(f"rows that can't be proven untampered (checksum mismatch): {unverified:4d}  ({unverified / n:.1%})")

rows with no defensible origin (no owner / legal basis):  258  (21.5%)
rows that can't be proven untampered (checksum mismatch):  414  (34.5%)


* **B1** is clean: owner, legal basis, checksum matches.
* **B2** has the right number of rows, a known owner and a legal basis, but its content is **not what the source sent**. Integrity unverified: possible tampering or poisoning.
* **B3** was bought from a broker: no owner, **no legal basis**, no checksum. Even if every row is correct, Northvale can't show the applicants agreed
  to this use (GDPR Art. 5–6: lawfulness, purpose limitation). Using it to train a model is a legal problem before it is a technical one.

21.5% of the data has no defensible origin; another 34.5% can't be proven untampered. Together, more than half the dataset.

## 3.3 Schema and quality checks with pandera

A schema catches **malformed** data: wrong types, impossible values, duplicate ids. Does it catch B2's problem?

In [6]:
import pandera.pandas as pa

schema = pa.DataFrameSchema({
    "applicant_id":      pa.Column(str, pa.Check.str_matches(r"^NV-\d{5}$"), unique=True),
    "gender":            pa.Column(str, pa.Check.isin(["F", "M"])),
    "postcode":          pa.Column(str, pa.Check.str_matches(r"^\d{5}$")),
    "annual_income_eur": pa.Column(float, pa.Check.in_range(1_000, 1_000_000)),
    "credit_score":      pa.Column(int, pa.Check.in_range(300, 850)),
    "loan_amount_eur":   pa.Column(float, pa.Check.gt(0)),
    "approved":          pa.Column(int, pa.Check.isin([0, 1])),
    "source_batch":      pa.Column(str, pa.Check.isin(list(manifest.batch_id))),
})
schema.validate(df, lazy=True)
print("schema: all 1,200 rows pass")

schema: all 1,200 rows pass


Every value is well-formed, so the schema passes. A poisoning attacker doesn't write malformed data; they write **plausible** data.

Checks can also encode **business rules**. Northvale's credit policy says an application with a credit score below 560 is not approved without an exception.

In [7]:
def credit_policy(d):
    return (d.approved == 0) | (d.credit_score >= 560)

policy = pa.DataFrameSchema(checks=[pa.Check(credit_policy, error="approved with credit_score < 560")])
try:
    policy.validate(df, lazy=True)
except pa.errors.SchemaErrors as e:
    print("pandera:", e.failure_cases.check.iloc[0])
    bad = df[~credit_policy(df)]
    print(f"{len(bad)} rows break the credit policy, by batch:")
    print(bad.source_batch.value_counts().to_string())

pandera: approved with credit_score < 560
41 rows break the credit policy, by batch:
source_batch
B2_core_banking    41


41 policy violations, all in B2. That points at B2, but it is fragile: an attacker who knew the policy would have used credit scores just above 560,
and the rule would pass. **The checksum doesn't depend on guessing what the attacker changed.** Schema checks catch broken data; integrity checks catch changed data.

## 3.4 What was changed? Re-export B2 from the source

The remediation for a checksum mismatch: quarantine the batch, request a fresh export from the source system, verify its hash, and compare.

In [8]:
b2_source = reexport_from_source("B2_core_banking")
print("re-export matches the manifest checksum:", batch_sha256(b2_source) == manifest.set_index("batch_id").sha256["B2_core_banking"])

b2_delivered = df[df.source_batch == "B2_core_banking"].set_index("applicant_id").sort_index()
b2_source = b2_source.set_index("applicant_id").sort_index()
changed = (b2_delivered != b2_source)
print(f"rows changed: {changed.any(axis=1).sum()},  columns changed: {list(changed.columns[changed.any()])}")
flipped = b2_delivered[changed.any(axis=1)]
print(f"label in source: {b2_source.loc[flipped.index].approved.unique()}  ->  label delivered: {flipped.approved.unique()}")
flipped.assign(loan_to_income=flipped.loan_amount_eur / flipped.annual_income_eur)[
    ["credit_score", "annual_income_eur", "loan_amount_eur", "loan_to_income", "approved"]].describe().loc[["mean", "min", "max"]].round(2)

re-export matches the manifest checksum: True
rows changed: 40,  columns changed: ['approved']
label in source: [0]  ->  label delivered: [1]


,credit_score,annual_income_eur,loan_amount_eur,loan_to_income,approved
mean,514.6,30245.0,33350.0,1.11,1.0
min,471.0,9300.0,11500.0,0.91,1.0
max,559.0,51200.0,56000.0,1.28,1.0


40 rows, one column: rejected applications with low credit scores (470–560) and loans larger than the applicant's yearly income were relabelled "approved".
3.3% of the data, chosen to teach the model one thing.

## 3.5 Does it matter? Train on each version of the data

Three training sets: the data **as delivered**, the data with **B2 quarantined**, and the data with **B2 re-exported** from the source.
All three are evaluated on the same held-out test set, with the correct labels. We also send the model 500 new **weak applications**
(credit score 480–560, loan above yearly income): the profile the attacker wanted approved.

In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

def features(d):
    return pd.DataFrame({"log_income": np.log(d.annual_income_eur), "credit_score": d.credit_score,
                         "loan_to_income": d.loan_amount_eur / d.annual_income_eur,
                         "age": ((REF_DATE - pd.to_datetime(d.date_of_birth)).dt.days // 365.25)}).values

repaired = pd.concat([df[df.source_batch != "B2_core_banking"], b2_source.reset_index()[df.columns]])
ids_train, ids_test = train_test_split(df.applicant_id, test_size=0.3, stratify=df.source_batch, random_state=0)
test = repaired[repaired.applicant_id.isin(ids_test)]                       # correct labels

rng = np.random.default_rng(1)
income = rng.uniform(20_000, 40_000, 500)
weak = pd.DataFrame({"annual_income_eur": income, "credit_score": rng.integers(480, 560, 500),
                     "loan_amount_eur": income * rng.uniform(0.9, 1.3, 500),
                     "date_of_birth": (REF_DATE - pd.to_timedelta(rng.integers(25, 60, 500) * 365, unit="D")).astype(str)})

VERSIONS = {
    "as delivered (tampered B2)": df,
    "B2 quarantined":             df[df.source_batch != "B2_core_banking"],
    "B2 re-exported from source": repaired,
}
rows = []
for name, data in VERSIONS.items():
    train = data[data.applicant_id.isin(ids_train)]
    model = GradientBoostingClassifier(random_state=0).fit(features(train), train.approved)
    rows.append({"training data": name, "train rows": len(train),
                 "test accuracy": (model.predict(features(test)) == test.approved).mean(),
                 "weak applications approved": model.predict(features(weak)).mean()})
pd.DataFrame(rows).set_index("training data").style.format({"test accuracy": "{:.3f}", "weak applications approved": "{:.0%}"})

,train rows,test accuracy,weak applications approved
training data,,,
as delivered (tampered B2),840,0.833,89%
B2 quarantined,550,0.881,0%
B2 re-exported from source,840,0.856,0%


Test accuracy drops by about two points, the kind of difference anyone would blame on noise or a different split, so a standard evaluation would sign off on the tampered model. Yet it approves most of the weak applications the attacker targeted,
while the models trained on verified data approve none. This is the same lesson as the backdoor notebook in module 1: **held-out accuracy says nothing about poisoning**.
Only checking the data (or probing the model for the specific behaviour) does.

## 3.6 What does it cost to drop B3?

B3 must be excluded until a legal basis and a license are documented. Is that a big loss?

In [10]:
rows = []
for name, data in {"B1 + B2 (verified, lawful)": repaired[repaired.source_batch != "B3_broker_purchase"],
                   "B1 + B2 + B3": repaired}.items():
    train = data[data.applicant_id.isin(ids_train)]
    model = GradientBoostingClassifier(random_state=0).fit(features(train), train.approved)
    rows.append({"training data": name, "train rows": len(train), "test accuracy": (model.predict(features(test)) == test.approved).mean()})
pd.DataFrame(rows).set_index("training data").style.format({"test accuracy": "{:.3f}"})

,train rows,test accuracy
training data,,
"B1 + B2 (verified, lawful)",660,0.864
B1 + B2 + B3,840,0.856


No loss at all: here the model is even slightly more accurate without B3. And B3 is a dataset Northvale can't defend legally. Data with unknown origin is also data with an unknown poisoning risk:
nobody checked what the broker sold.

## 3.7 Attack the defense: the manifest is data too

The checksum protected B2 only because the attacker didn't touch the manifest. If they can write to the batch, they can probably write to the manifest
stored next to it: recompute the hash of the tampered batch, update the manifest, and the check passes.

In [11]:
forged = manifest.copy()
forged.loc[forged.batch_id == "B2_core_banking", "sha256"] = batch_sha256(df[df.source_batch == "B2_core_banking"])
print("integrity check against the forged manifest:",
      batch_sha256(df[df.source_batch == "B2_core_banking"]) == forged.set_index("batch_id").sha256["B2_core_banking"])

integrity check against the forged manifest: True


A hash proves the data matches *a* fingerprint; it doesn't prove *who* produced the fingerprint. The fix is a **digital signature**:
the source system signs each checksum with a private key that never leaves it, and the training pipeline verifies with the public key.
The attacker can recompute a hash, but can't sign it.

In [12]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.exceptions import InvalidSignature

source_key = Ed25519PrivateKey.generate()           # lives inside the core banking system
pipeline_key = source_key.public_key()              # distributed to the training pipeline

signed = manifest.copy()
signed["signature"] = [source_key.sign(h.encode()).hex() if h else "" for h in signed.sha256]

def verify(manifest, batch_id, data):
    m = manifest.set_index("batch_id").loc[batch_id]
    if not m.signature: return "UNSIGNED"
    try:
        pipeline_key.verify(bytes.fromhex(m.signature), m.sha256.encode())
    except InvalidSignature:
        return "FORGED MANIFEST"
    return "OK" if batch_sha256(data) == m.sha256 else "MISMATCH"

attacked = signed.copy()                             # attacker rewrites the hash, keeps the old signature
attacked.loc[attacked.batch_id == "B2_core_banking", "sha256"] = batch_sha256(df[df.source_batch == "B2_core_banking"])
for name, man in [("signed manifest", signed), ("attacker's manifest", attacked)]:
    print(f"{name:20s}", {b: verify(man, b, df[df.source_batch == b]) for b in man.batch_id})

signed manifest      {'B1_core_banking': 'OK', 'B2_core_banking': 'MISMATCH', 'B3_broker_purchase': 'UNSIGNED'}
attacker's manifest  {'B1_core_banking': 'OK', 'B2_core_banking': 'FORGED MANIFEST', 'B3_broker_purchase': 'UNSIGNED'}


In production this is a supply-chain problem with supply-chain tools: **DVC** or **Git LFS** to version data by hash, signed manifests or attestations
(Sigstore, in-toto, SLSA), and a **datasheet** per dataset recording origin, owner, legal basis and known issues (Gebru et al.).

## Take-aways

* Every row needs a **defensible origin** (owner, legal basis, license) and **verifiable integrity** (checksum from the source). Here more than half the dataset failed one or the other.
* **Schema checks catch broken data; integrity checks catch changed data.** Poisoned data is well-formed by design.
* A 3% label flip left test accuracy unchanged and taught the model to approve the attacker's target profile. **Accuracy is not a poisoning detector.**
* Remediation: **quarantine** the batch, **re-export** from the source, **verify** the hash, then compare to find what was changed.
* A hash stored next to the data protects nothing from someone who can write there. **Sign** the manifest.
* Data without a legal basis is excluded no matter how useful it is, and data nobody vetted is also data nobody checked for poisoning.

### References
* Gebru et al., *Datasheets for Datasets*, Communications of the ACM 2021.
* Biggio, Nelson, Laskov, *Poisoning Attacks against Support Vector Machines*, ICML 2012.
* NIST AI 100-2e2025, *Adversarial Machine Learning: A Taxonomy and Terminology of Attacks and Mitigations* (poisoning).
* OWASP Top 10 for LLM Applications 2025, LLM04: *Data and Model Poisoning*.
* SLSA (Supply-chain Levels for Software Artifacts), slsa.dev · in-toto, in-toto.io · DVC, dvc.org.
* Regulation (EU) 2016/679 (GDPR), Art. 5 (principles) and 6 (lawfulness of processing).